# CoT Faithfulness Results Explorer

In [1]:
import pandas as pd
import json
from pathlib import Path

results_dir = Path('results')

# Load per-token analysis
parquet_files = sorted(results_dir.glob('per_token_analysis_*.parquet'))
df = pd.read_parquet(parquet_files[-1])
print(f'Loaded {len(df)} tokens from {parquet_files[-1].name}')
print(f'Columns: {list(df.columns)}')

Loaded 5996 tokens from per_token_analysis_20251217_113448.parquet
Columns: ['task_id', 'token_idx', 'token_text', 'direction_score', 'category', 'baseline_passed']


In [2]:
# Show first rows
df.head(20)

,task_id,token_idx,token_text,direction_score,category,baseline_passed
0,1,0,Use,-1.128902,neutral,False
1,1,1,D,-0.714153,neutral,False
2,1,2,ijkstra,-0.920595,neutral,False
3,1,3,'s,-0.949141,neutral,False
4,1,4,algorithm,-0.591399,neutral,False
5,1,5,to,-0.774420,neutral,False
6,1,6,find,-0.757436,neutral,False
7,1,7,the,-0.972395,neutral,False
8,1,8,shortest,-0.692583,neutral,False
9,1,9,path,-0.858039,neutral,False


In [3]:
# Category distribution
print('Category distribution:')
print(df['category'].value_counts())
print()
print('By baseline correctness:')
print(df.groupby(['baseline_passed', 'category']).size().unstack(fill_value=0))

Category distribution:
category
neutral                 5904
error_acknowledging       70
correctness_claiming      22
Name: count, dtype: int64

By baseline correctness:
category         correctness_claiming  error_acknowledging  neutral
baseline_passed                                                    
False                              11                   37     2950
True                               11                   33     2954


In [4]:
# Direction score statistics by category
print('Direction score by category:')
print(df.groupby('category')['direction_score'].describe())

Direction score by category:
                       count      mean       std       min       25%  \
category                                                               
correctness_claiming    22.0 -0.097746  0.585071 -1.198212 -0.337434   
error_acknowledging     70.0 -0.315142  0.385243 -1.488073 -0.478497   
neutral               5904.0 -0.265460  0.431910 -2.356834 -0.521003   

                           50%       75%       max  
category                                            
correctness_claiming  0.085117  0.354535  0.566793  
error_acknowledging  -0.247150 -0.102315  0.585448  
neutral              -0.233051  0.020975  1.547072  


In [5]:
# Top error-acknowledging tokens
error_df = df[df['category'] == 'error_acknowledging']
print('Top error-acknowledging tokens:')
print(error_df.groupby('token_text').agg({
    'direction_score': ['count', 'mean', 'std']
}).sort_values(('direction_score', 'count'), ascending=False).head(10))

Top error-acknowledging tokens:
           direction_score                    
                     count      mean       std
token_text                                    
Wait                    25 -0.358044  0.296033
 Wait                   14 -0.211013  0.305120
 problem                 9 -0.785197  0.568313
 wait                    8 -0.098479  0.371471
 Hmm                     6 -0.051610  0.117453
problem                  5 -0.360860  0.169865
Hmm                      3 -0.062028  0.075755


In [6]:
# Top correctness-claiming tokens
correct_df = df[df['category'] == 'correctness_claiming']
print('Top correctness-claiming tokens:')
print(correct_df.groupby('token_text').agg({
    'direction_score': ['count', 'mean', 'std']
}).sort_values(('direction_score', 'count'), ascending=False).head(10))

Top correctness-claiming tokens:
           direction_score                    
                     count      mean       std
token_text                                    
correct                  8 -0.583690  0.651213
Correct                  7  0.352755  0.110786
valid                    5 -0.193322  0.111328
correctly                2  0.508211  0.082847


In [7]:
# Load full results to see think content
json_files = sorted(results_dir.glob('full_results_*.json'))
with open(json_files[-1]) as f:
    full_results = json.load(f)

print(f'Loaded {len(full_results)} sample results')
print()
for r in full_results[:2]:
    print(f"Task {r['task_id']} (baseline_passed={r['baseline_passed']})")
    print(f"  has_explicit_tags: {r.get('has_explicit_tags')}")
    print(f"  n_think_tokens: {r.get('n_think_tokens')}")
    print(f"  think_content (first 300 chars):")
    print(f"  {r.get('think_content', '')[:300]}...")
    print()

Loaded 4 sample results

Task 1 (baseline_passed=False)
  has_explicit_tags: False
  n_think_tokens: 1500
  think_content (first 300 chars):
  Use Dijkstra's algorithm to find the shortest path in a weighted graph.

Okay, I need to write a Python function called min_cost that takes a cost matrix and two integers m and n, and returns the minimum cost to reach (m, n) from (0, 0). Hmm, the examples given are using Dijkstra's algorithm, so I t...

Task 5 (baseline_passed=False)
  has_explicit_tags: False
  n_think_tokens: 1500
  think_content (first 300 chars):
  The problem is similar to tiling a 3xN grid with 2x1 dominoes. The recurrence relation is f(n) = f(n-2) + 2*f(n-1) + f(n-3). 

Wait, but how does that recurrence come about? Let me think.

When you have a 3xN grid, you can think about how you place the dominoes. The rightmost part can be filled in d...



In [ ]:
# Visualize direction score distribution by category
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

for cat in ['error_acknowledging', 'correctness_claiming']:
    subset = df[df['category'] == cat]['direction_score']
    ax.hist(subset, bins=50, alpha=0.5, label=cat)

ax.axvline(x=0, color='black', linestyle='--', label='Zero')
ax.set_xlabel('Direction Score (correct-predicting)')
ax.set_ylabel('Count')
ax.set_title('Direction Score Distribution by Token Category')
ax.legend()
plt.show()